# 02 · Chunking experiment (Module 3)

Chunk size trades **precision** (small chunks, tight context) against **recall** (large chunks, more context per hit).

Rather than defaulting to 500 tokens, sweep it and measure.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
from pathlib import Path

from src.ingestion.chunker import ChunkConfig, chunk_documents
from src.ingestion.loader import iter_documents

docs = list(iter_documents(Path("../data/raw")))
SIZES = [300, 500, 800, 1200]

stats = []
chunk_sets = {}
for size in SIZES:
    cfg = ChunkConfig(target_tokens=size, overlap_tokens=int(size * 0.15))
    chunks = chunk_documents(docs, cfg)
    chunk_sets[size] = chunks
    tokens = pd.Series([c.token_count for c in chunks])
    stats.append({
        "target": size,
        "overlap": cfg.overlap_tokens,
        "chunks": len(chunks),
        "mean_tokens": tokens.mean(),
        "median_tokens": tokens.median(),
        "p95_tokens": tokens.quantile(0.95),
    })

stats = pd.DataFrame(stats).round(1)
stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(stats["target"], stats["chunks"], "o-", color="#4C9BE8")
axes[0].set(xlabel="target tokens", ylabel="chunks", title="Index size vs chunk size")
for size in SIZES:
    pd.Series([c.token_count for c in chunk_sets[size]]).plot.kde(ax=axes[1], label=f"{size}")
axes[1].set(xlabel="tokens per chunk", title="Realised chunk size", xlim=(0, 1500))
axes[1].legend()
plt.tight_layout()

Realised size sits below target because chunks stop at sentence and section boundaries rather than mid-claim.

## Retrieval quality per chunk size\n\nThis is the number that actually matters. Requires an evaluation set (`hcrag make-evalset`).\n\nScored at **document** granularity: chunk IDs are a function of the chunking, so gold chunk labels cannot survive a change in chunk size.

In [ ]:
from src.evaluation.experiments import run_chunk_sweep

results = run_chunk_sweep(SIZES)          # also available as: hcrag sweep
sweep = pd.DataFrame(results.rows)
sweep.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for metric in ["recall@5", "recall@20", "mrr"]:
    axes[0].plot(sweep["target_tokens"], sweep[metric], "o-", label=metric)
axes[0].set(xlabel="chunk tokens", ylabel="score", title="Retrieval quality")
axes[0].legend()

axes[1].plot(sweep["target_tokens"], sweep["query_ms"], "o-", color="#C0504D")
axes[1].set(xlabel="chunk tokens", ylabel="ms", title="Query latency")
plt.tight_layout()

In [ ]:
best = sweep.loc[sweep["recall@5"].idxmax()]
print(f"Best Recall@5: {best['target_tokens']:.0f} tokens → {best['recall@5']:.3f}")
print(f"Index size at that setting: {best['n_chunks']:.0f} chunks")